# 07 · Label-assist model (small dataset → Roboflow auto-label)

Trains a segmentation model on the hand-corrected images in Roboflow project **`ugalab/powdery-mildew-mbjv6`**,
then uploads its weights back to that Roboflow version so **Label Assist** can pre-annotate the remaining images.

Loop: correct ~70 images → train here → upload → Label Assist the next batch → correct → new Roboflow version → set
`VERSION` and run again. Each round has more data, so the model (and its pre-labels) get better.

**Model choice for ~70 images:** YOLO26-seg, fine-tuned **from the v1 hydrangea model**
(`runs/yolo26x-seg_data-v1/weights/best.pt`) instead of COCO. It already knows these leaves and lesions, so a small
dataset only has to correct it; starting from COCO with 70 images would mostly learn from scratch. Roboflow accepts
custom YOLO26 segmentation weights (`model_type = "yolo26x-seg"`), so the result plugs straight into Label Assist.

In [ ]:
import os
import shutil
from pathlib import Path

import torch
import yaml
from dotenv import load_dotenv
from PIL import Image

ROOT = Path.cwd()
load_dotenv(ROOT / ".env")  # ROBOFLOW_API_KEY lives here (gitignored); never paste the key into a notebook
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Settings

- `VERSION`: the Roboflow dataset version to train on. Bump it each round. Weights are uploaded to this version.
- `FORMAT = "yolo26"`: the Roboflow export used by Ultralytics YOLO26 (same layout as `yolov8`/`yolo11`).
- `INIT = "v1"` fine-tunes the v1 hydrangea model (size fixed to `x`, its size). `INIT = "coco"` starts from COCO
  weights of size `YOLO_SIZE` (use it if the class names here differ from v1, or for a smaller/faster model).
- Small data → many epochs with a long patience, strong geometric augmentation, and `copy_paste` (pastes lesions
  between images, which helps the rare mildew class). Colour augmentation stays mild: colour separates the classes.
- `IMGSZ = None` uses the dataset's own image size (the Roboflow resize), which is also what Label Assist sends.
- `OVERLAP_MASK = False`: each mask is learned whole (the default `True` teaches leaves with holes at the lesions).

In [ ]:
WORKSPACE = "ugalab"
PROJECT = "powdery-mildew-mbjv6"
VERSION = 2
FORMAT = "yolo26"

INIT = "v1"  # "v1" (fine-tune the hydrangea model) or "coco"
V1_WEIGHTS = ROOT / "runs" / "yolo26x-seg_data-v1" / "weights" / "best.pt"
YOLO_SIZE = "x"  # used with INIT = "coco": n, s, m, l, x
RUN_TAG = ""  # e.g. "_try2" for a separate run

EPOCHS = 200
PATIENCE = 60
IMGSZ = None  # None → dataset image size
BATCH = 8  # x at 1024 ≈ 24 GB; lower it (or -1 for auto) on a smaller GPU
OVERLAP_MASK = False
AUG = dict(
    hsv_h=0.0, hsv_s=0.3, hsv_v=0.3,
    degrees=20, translate=0.1, scale=0.5, shear=2.0,
    fliplr=0.5, flipud=0.5,
    mosaic=1.0, close_mosaic=20, mixup=0.0, copy_paste=0.3,
)

if INIT == "v1" and not V1_WEIGHTS.exists():
    print(f"{V1_WEIGHTS} not found → using COCO weights")
    INIT = "coco"
if INIT == "v1":
    YOLO_SIZE = "x"

DATA_NAME = f"{PROJECT}-v{VERSION}"
DATA_DIR = ROOT / "datasets" / f"{DATA_NAME}-{FORMAT}"
RUN_NAME = f"labelassist-yolo26{YOLO_SIZE}-seg-{INIT}_data-{DATA_NAME}{RUN_TAG}"
RUN_DIR = ROOT / "runs" / RUN_NAME
MODEL_TYPE = f"yolo26{YOLO_SIZE}-seg"  # Roboflow model_type for the upload
print(RUN_NAME)

## Download the dataset (skipped if already downloaded)

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
version = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION)
if not (DATA_DIR / "data.yaml").exists():
    version.download(FORMAT, location=str(DATA_DIR))
print(DATA_DIR)

## Check the dataset

Counts images and instances per split and class. Roboflow writes relative paths into `data.yaml` that don't always
resolve, so a `data_local.yaml` with absolute paths is written next to it. With very few images Roboflow may give no
validation split; then training images are used for validation too (the metrics will be optimistic, but training
still works). Better: in Roboflow, put ~10–15 % of the images in **valid** before generating the version.

In [ ]:
import collections

src = yaml.safe_load(open(DATA_DIR / "data.yaml"))
names = src["names"] if isinstance(src["names"], list) else [src["names"][k] for k in sorted(src["names"])]
print("classes:", names)


def split_stats(split):
    imgs = sorted((DATA_DIR / split / "images").glob("*")) if (DATA_DIR / split / "images").exists() else []
    counts = collections.Counter()
    for lbl in (DATA_DIR / split / "labels").glob("*.txt"):
        for line in lbl.read_text().splitlines():
            if line.strip():
                counts[names[int(line.split()[0])]] += 1
    return imgs, counts


splits = {}
for split in ("train", "valid", "test"):
    imgs, counts = split_stats(split)
    splits[split] = imgs
    print(f"{split:<6} {len(imgs):>4} images | " + ", ".join(f"{n}: {counts[n]}" for n in names))

val_split = "valid" if splits["valid"] else "train"
if val_split == "train":
    print("WARNING: no validation images → validating on train (optimistic metrics)")

data_cfg = {"path": str(DATA_DIR), "train": "train/images", "val": f"{val_split}/images", "names": names}
if splits["test"]:
    data_cfg["test"] = "test/images"
DATA_YAML = DATA_DIR / "data_local.yaml"
yaml.safe_dump(data_cfg, open(DATA_YAML, "w"), sort_keys=False)

if IMGSZ is None:
    w, h = Image.open(splits["train"][0]).size
    IMGSZ = min(1024, (max(w, h) + 31) // 32 * 32)
print("image size:", IMGSZ)

if INIT == "v1":
    from ultralytics import YOLO

    v1_names = list(YOLO(V1_WEIGHTS).names.values())
    if v1_names == names:
        print("class names match v1 → the whole model (including the class head) is fine-tuned")
    else:
        print(f"class names differ from v1 {v1_names} → the backbone is reused and the head is relearned")

## Train

Resumes automatically like `02_train.ipynb`: no checkpoint → new run; unfinished `weights/last.pt` → resume;
finished → nothing to do (use `RUN_TAG` for another run). `workers=0` avoids Jupyter data-loader hangs on Windows.

In [ ]:
from ultralytics import YOLO

LAST = RUN_DIR / "weights" / "last.pt"
if not LAST.exists():
    ACTION = "new"
else:
    ckpt = torch.load(LAST, map_location="cpu", weights_only=False)
    ACTION = "finished" if ckpt.get("optimizer") is None else "resume"
    del ckpt
print(ACTION, "→", RUN_DIR)

if ACTION == "resume":
    YOLO(LAST).train(resume=True)
elif ACTION == "new":
    model = YOLO(V1_WEIGHTS if INIT == "v1" else f"yolo26{YOLO_SIZE}-seg.pt")
    model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        patience=PATIENCE,
        imgsz=IMGSZ,
        batch=BATCH,
        overlap_mask=OVERLAP_MASK,
        project=str(ROOT / "runs"),
        name=RUN_NAME,
        exist_ok=True,
        device=0,
        workers=0,
        seed=0,
        **AUG,
    )

## Validate

Mask mAP per class on the validation split (test split too, if there is one). With ~10 validation images these
numbers are rough; the picture check below matters as much.

In [ ]:
best = YOLO(RUN_DIR / "weights" / "best.pt")
for split in ["val"] + (["test"] if splits["test"] else []):
    m = best.val(data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=4, workers=0, verbose=False, plots=False)
    print(f"\n{split}: mask mAP50 {m.seg.map50:.3f}  mAP50-95 {m.seg.map:.3f} | box mAP50 {m.box.map50:.3f}")
    for i, c in enumerate(m.seg.ap_class_index):
        print(f"  {names[c]:<15} mask mAP50 {m.seg.ap50[i]:.3f}  mAP50-95 {m.seg.ap[i]:.3f}")

## Look at predictions

A few validation images with predicted masks. `CONF` here only affects these pictures; in Roboflow you pick
the Label Assist confidence in the UI (start around 0.3–0.4 and lower it if lesions are missed).

In [ ]:
from IPython.display import display

CONF = 0.35
for p in splits[val_split][:6]:
    r = best.predict(p, conf=CONF, imgsz=IMGSZ, retina_masks=True, verbose=False)[0]
    im = Image.fromarray(r.plot(boxes=False)[..., ::-1])
    im.thumbnail((900, 900))
    print(p.name, dict(collections.Counter(r.names[int(c)] for c in r.boxes.cls)))
    display(im)

## Upload to Roboflow for Label Assist

Uploads `weights/best.pt` to **version `VERSION`** of the project (the SDK reads classes and training settings
from the checkpoint and accepts any recent Ultralytics for `yolo26*`).
Set `UPLOAD = True` to run it. Roboflow processes the weights for a few minutes; afterwards the model is listed
under the version's **Models** tab.

Then in Roboflow: open an unannotated image in **Annotate** → **Label Assist** (magic-wand icon) → pick this
model → choose the classes and confidence → it draws the masks; you correct them. Batch use: **Auto Label** on a
batch of images with the same model.

If the upload or the hosted preview fails for `yolo26x-seg`, the fallback is `INIT = "coco"`, `YOLO_SIZE = "l"` or
`"m"` (smaller upload), or the `rfdetr-seg-*` types Roboflow also accepts.

In [ ]:
UPLOAD = False

if UPLOAD:
    version.deploy(model_type=MODEL_TYPE, model_path=str(RUN_DIR), filename="weights/best.pt")
else:
    print(f"set UPLOAD = True to upload {RUN_DIR / 'weights' / 'best.pt'} as {MODEL_TYPE} to {PROJECT} v{VERSION}")

## Next round

1. Correct the Label Assist annotations for the next batch of images in Roboflow.
2. **Generate** a new version (no augmentation needed for this loop; keep ~10–15 % in valid).
3. Set `VERSION` to it and run this notebook again. `INIT = "v1"` still starts from the v1 model; to build on the
   previous round instead, point `V1_WEIGHTS` at that round's `runs/labelassist-…/weights/best.pt`.